In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from src.analysis_utils import (
    drop_missing,
    subset_variables,
    recode_vaccination_variables,
    chi2_cramers_v_weighted,
    normalized_crosstab,
    screen_categorical_associations,
    fit_weighted_logit,
    fit_weighted_ordinal_logit,
    extract_odds_ratios,
    plot_heatmap,
    plot_odds_ratios,
    plot_weighted_coverage_bar,
    plot_stacked_vaccination
)

In [2]:
# Path relative to script.py
csv_path = "../data/cleaned_data_NA_included.csv"

# Read CSV
df = pd.read_csv(csv_path)

# Display first 5 rows
df.head()

,Mothers_id,Birth_Order,Cluster_id,Household_id,v003,v005,Childs_Age,Received_Hep_B_at_birth,Hep_B_1,Hep_B_2,...,hhid,Number_of_household_members,Number_of_children_under_5,Result_of_household_interview,Region,Type_of_place_of_residence,Translator_used,Vaccination at Birth,Vaccination Flag,wt
0,457 11 2,1,457,11,2,603700,7,No,No,No,...,457 11,3,1,Completed,Ratanak Kiri,Rural,No,False,None Post Birth,0.603700
1,455 4 2,2,455,4,2,229619,11,Vaccination date on card,No,No,...,455 4,4,1,Completed,Ratanak Kiri,Rural,No,True,None Post Birth,0.229619
2,455 17 2,1,455,17,2,229619,12,No,No,No,...,455 17,4,1,Completed,Ratanak Kiri,Rural,No,False,None Post Birth,0.229619
3,304 4 4,1,304,4,4,121341,10,No,No,No,...,304 4,6,1,Completed,Mondul Kiri,Urban,No,False,None Post Birth,0.121341
4,303 27 2,4,303,27,2,131714,22,No,No,No,...,303 27,5,1,Completed,Mondul Kiri,Rural,No,False,None Post Birth,0.131714


In [3]:
df = df.rename(columns={
    'Vaccination Flag': 'Vaccination_Flag',
    'Vaccination at Birth': 'Vaccination_at_Birth'
})
df["Respondent_country_of_birth"] = df["Respondent_country_of_birth"].astype(str)

In [4]:
from scipy import stats

# --- MAR TEST ---
df_temp = df.copy()
df_temp["is_missing"] = df_temp["Received_Hep_B_at_birth"].isnull().astype(int)

exclude = [
    "is_missing", "wt",
    "Received_Hep_B_at_birth", "Hep_B_1", "Hep_B_2", "Hep_B_3",
    "Vaccination_Flag", "Vaccination_at_Birth"
]

predictors = [col for col in df_temp.columns if col not in exclude]
results = []

for var in predictors:
    try:
        if df_temp[var].nunique() <= 1:
            continue

        if df_temp[var].dtype in ["float64", "int64"] and df_temp[var].nunique() >= 10:
            g0 = df_temp[df_temp["is_missing"] == 0][var].dropna()
            g1 = df_temp[df_temp["is_missing"] == 1][var].dropna()
            if len(g0) > 1 and len(g1) > 1:
                _, p = stats.ttest_ind(g0, g1, equal_var=False)
                test_used = "Welch's T-Test"
            else:
                continue
        else:
            result = chi2_cramers_v_weighted(df_temp, var, "is_missing", weight_col="wt")
            p = result["p_value"]
            test_used = "Chi-Square"

        results.append({"Variable": var, "Test": test_used, "P_Value": p})
    except Exception:
        continue

report = pd.DataFrame(results)
report["Significant"] = report["P_Value"] < 0.05
report = report.sort_values("P_Value")

print("--- MISSINGNESS ANALYSIS: Received_Hep_B_at_birth ---")
print(report[report["Significant"]].to_string(index=False))

--- MISSINGNESS ANALYSIS: Received_Hep_B_at_birth ---
                                   Variable           Test      P_Value  Significant
                                 Childs_Age Welch's T-Test 0.000000e+00         True
              Timing_of_1st_antenatal_check     Chi-Square 0.000000e+00         True
Number_of_antenatal_visits_during_pregnancy Welch's T-Test 0.000000e+00         True
                                Mothers_age Welch's T-Test 2.133439e-48         True
                                       v003 Welch's T-Test 1.142046e-17         True
                         Mothers_occupation     Chi-Square 1.837137e-15         True
                Number_of_household_members Welch's T-Test 1.463353e-07         True
                 Number_of_children_under_5     Chi-Square 1.914737e-04         True
                  Highest_educational_level     Chi-Square 6.729056e-03         True
                   Household_has_television     Chi-Square 7.812367e-03         True
           

In [6]:
# --- DROP MISSING BLOCK ---
df_clean = drop_missing(df, columns=["Received_Hep_B_at_birth"])
print(f"Rows before drop: {len(df)}")
print(f"Rows after drop:  {len(df_clean)}")
print(f"Rows removed:     {len(df) - len(df_clean)}")

Rows before drop: 8153
Rows after drop:  4887
Rows removed:     3266


In [7]:
# --- Inconsistent DHS Code Check ---
inconsistent_codes = [97, 997, 9997]

# Find columns containing inconsistent codes
inconsistent_cols = [
    col for col in df.columns
    if df[col].dtype in ["float64", "int64"]
    and df[col].isin(inconsistent_codes).any()
]

# Filter rows with at least one inconsistent code
inconsistent_df = df[df[inconsistent_cols].isin(inconsistent_codes).any(axis=1)]

print(f"Rows with inconsistent codes (97, 997, 9997): {len(inconsistent_df)}")
print("\nColumns where these occur:")
print(df[inconsistent_cols].isin(inconsistent_codes).sum().sort_values(ascending=False))

inconsistent_df[inconsistent_cols].head()

Rows with inconsistent codes (97, 997, 9997): 14

Columns where these occur:
Cluster_id    14
dtype: int64


,Cluster_id
160,97
5875,97
5876,97
5877,97
5878,97


In [8]:
def prepare_brant_df(df):
    d = df.copy()

    edu_map = {
        "No education": "No education",
        "Primary": "Primary",
        "Secondary": "Secondary",
        "Higher": "Higher"
    }
    d["Education"] = d["Highest_educational_level"].map(edu_map).fillna("No education")

    wealth_map = {
        "Poorest": "Poorest",
        "Poorer": "Poor",
        "Middle": "Middle",
        "Richer": "Rich",
        "Richest": "Richest"
    }
    d["Wealth"] = d["Wealth_index_combined"].map(wealth_map).fillna("Middle")

    lit_map = {
        "Cannot read at all": "Non-Reader",
        "Blind/visually impaired": "Non-Reader",
        "No card with required language": "Non-Reader",
        "Able to read only parts of a sentence": "Partial",
        "Able to read whole sentence": "Full"
    }
    d["Literacy"] = d["Literacy"].map(lit_map).fillna("Non-Reader")

    d["Maternal_Age_Cat"] = pd.cut(
        d["Mothers_age"],
        bins=[0, 24, 35, 100],
        labels=["< 24", "24-35", "> 35"]
    )

    hep_map = {
        "Vaccination date on card": 1,
        "Vaccination marked on card": 1,
        "Reported by mother": 1,
        "No": 0,
        "Don't know": 0
    }
    d["HepB_Binary"] = d["Received_Hep_B_at_birth"].map(hep_map)
    d = d.sort_values(["Mothers_id", "Birth_Order"])
    d["Prev_Stat"] = d.groupby("Mothers_id")["HepB_Binary"].shift(1)
    d["History_Cat"] = pd.Categorical(
        d["Prev_Stat"].map(
            lambda x: "0_No_Prev_Data" if pd.isnull(x)
            else ("2_Prev_Child_Yes" if x == 1 else "1_Prev_Child_No")
        ),
        categories=["0_No_Prev_Data", "1_Prev_Child_No", "2_Prev_Child_Yes"]
    )

    d["Region_Name"] = d["Region"].astype(str)

    d["Education"] = pd.Categorical(
        d["Education"],
        categories=["No education", "Primary", "Secondary", "Higher"]
    )
    d["Literacy"] = pd.Categorical(
        d["Literacy"],
        categories=["Non-Reader", "Partial", "Full"]
    )
    d["Wealth"] = pd.Categorical(
        d["Wealth"],
        categories=["Poorest", "Poor", "Middle", "Rich", "Richest"]
    )

    return d

In [12]:
from src.analysis_utils import recode_vaccination_variables, fit_weighted_logit
import numpy as np
import pandas as pd

# --- 1. Data Prep ---
df_ready = prepare_brant_df(df_clean)
df_ready = recode_vaccination_variables(df_ready)
df_ready = df_ready.rename(columns={"Vaccination_Flag_ord": "vax_ord"})

# --- 2. Convert categoricals to strings to avoid ordered factor issue in R ---
df_brant = df_ready.copy()
for col in ["Maternal_Age_Cat", "Education", "Literacy", "Wealth", "History_Cat", "Region_Name"]:
    df_brant[col] = df_brant[col].astype(str)

# --- 3. Brant Test ---
variables_to_test = [
    "Maternal_Age_Cat",
    "Education",
    "Literacy",
    "Wealth",
    "History_Cat",
    "Number_of_antenatal_visits_during_pregnancy",
    "Region_Name"
]

all_betas = []
for var in variables_to_test:
    print(f"Processing: {var}")
    temp_results = {}
    for thresh in [0, 1, 2]:
        df_brant[f"gt_{thresh}"] = (df_brant["vax_ord"] > thresh).astype(int)
        try:
            result, _ = fit_weighted_logit(
                df_brant,
                formula=f"gt_{thresh} ~ {var}",
                weight_col="wt"
            )
            for name, row in result.iterrows():
                if "(Intercept)" in name:
                    continue
                if name not in temp_results:
                    temp_results[name] = {}
                temp_results[name][f"Cut_{thresh}"] = np.log(row["OR"])
        except Exception as e:
            print(f"  Failed on Cut {thresh}: {e}")
            continue
    if temp_results:
        var_df = pd.DataFrame(temp_results).T
        all_betas.append(var_df)

# --- 4. Consolidation & Range ---
master_brant = pd.concat(all_betas)
master_brant["Range"] = master_brant.max(axis=1) - master_brant.min(axis=1)

# --- 5. Report ---
print("\n--- BRANT TEST: COEFFICIENTS ACROSS THRESHOLDS ---")
print(f"{'Variable':<60} {'Cut_0':>8} {'Cut_1':>8} {'Cut_2':>8} {'Range':>8}")
print("-" * 92)
for idx, row in master_brant.round(4).iterrows():
    cut0 = f"{row.get('Cut_0', float('nan')):>8.4f}"
    cut1 = f"{row.get('Cut_1', float('nan')):>8.4f}"
    cut2 = f"{row.get('Cut_2', float('nan')):>8.4f}"
    rang = f"{row['Range']:>8.4f}"
    print(f"{idx:<60} {cut0} {cut1} {cut2} {rang}")

print("\n--- VARIABLES WITH LARGE COEFFICIENT RANGE (>0.5) ---")
flagged = master_brant[master_brant["Range"] > 0.5]
if len(flagged) == 0:
    print("None — proportional odds assumption holds for all variables.")
else:
    print(flagged.round(4).to_string())

Processing: Maternal_Age_Cat
Processing: Education
Processing: Literacy
Processing: Wealth
Processing: History_Cat
Processing: Number_of_antenatal_visits_during_pregnancy
Processing: Region_Name

--- BRANT TEST: COEFFICIENTS ACROSS THRESHOLDS ---
Variable                                                        Cut_0    Cut_1    Cut_2    Range
--------------------------------------------------------------------------------------------
Maternal_Age_Cat> 35                                           0.3971   0.2810   0.2887   0.1162
Maternal_Age_Cat24-35                                          0.5703   0.4865   0.5197   0.0838
EducationNo education                                         -1.1040  -0.8926  -0.8914   0.2126
EducationPrimary                                              -0.2345  -0.1556  -0.3125   0.1570
EducationSecondary                                             0.0194   0.0088  -0.0279   0.0473
LiteracyNon-Reader                                            -0.9570  -0.7240